# 04 - Classification & Regression
## Đề tài 11: Phân tích đánh giá khách sạn

**Mục tiêu:**
- Sentiment Classification (Baseline + Strong Model)
- Rating Regression
- F1-macro, MAE, RMSE metrics

In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from src.data.loader import load_data
from src.data.cleaner import DataCleaner
from src.features.builder import FeatureBuilder
from src.models.supervised import SentimentClassifier
from src.models.regression import RatingPredictor
from sklearn.model_selection import train_test_split

[WARNING] HDBSCAN not available. Install with: pip install hdbscan


## 1. Prepare Data

In [2]:
df = load_data(n_rows=2000)
cleaner = DataCleaner()
df_cleaned, _ = cleaner.clean(df)

# Create features
feature_builder = FeatureBuilder()
cleaned_texts = df_cleaned['cleaned_text'].astype(str).tolist()
tfidf_matrix = feature_builder.build_tfidf_features(cleaned_texts, fit=True)
stat_features = feature_builder.build_statistical_features(df_cleaned)

# Combine features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
stat_scaled = scaler.fit_transform(stat_features)
X = np.hstack([tfidf_matrix.toarray(), stat_scaled])

# Labels
y_sentiment = df_cleaned['sentiment'].values
y_rating = df_cleaned['rating'].values

# Split data
X_train, X_test, y_train_sent, y_test_sent = train_test_split(X, y_sentiment, test_size=0.2, random_state=42)
X_train, X_test, y_train_reg, y_test_reg = train_test_split(X, y_rating, test_size=0.2, random_state=42)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

[WARNING] Config file not found at configs\params.yaml
Using default configuration...
[INFO] File not found at data\raw\hotel_reviews.csv
[INFO] Generating sample hotel reviews data...
[INFO] Generating 2000 sample hotel reviews...
[INFO] Generated 2000 sample reviews
[INFO] Rating distribution:
rating
1    147
2    208
3    579
4    477
5    589
Name: count, dtype: int64
DATA CLEANING PROCESS

[Step 1/5] Basic data cleaning...
  - Original rows: 2000

[Step 2/5] Handling missing values...

[Step 3/5] Text preprocessing...
[INFO] Cleaning text...

[Step 4/5] Validating ratings...

[Step 5/5] Adding derived features...

CLEANING SUMMARY
Original rows: 2000
Final rows: 2000
Rows removed: 0 (0.0%)
Columns: ['review_text', 'rating', 'sentiment', 'reviewer_name', 'date', 'hotel_name', 'original_text', 'original_length', 'original_word_count', 'cleaned_text', 'cleaned_length', 'cleaned_word_count', 'length_reduction', 'word_reduction', 'review_length', 'word_count', 'avg_word_length', 'sente

## 2. Sentiment Classification

In [3]:
# Initialize classifier
classifier = SentimentClassifier()

# Train baselines
classifier.train_baselines(X_train, y_train_sent)

# Train strong model
classifier.train_strong_model(X_train, y_train_sent)

# Evaluate
clf_results = classifier.evaluate(X_test, y_test_sent)

# Compare models
comparison = classifier.compare_models()
print("\nModel Comparison:")
print(comparison)


TRAINING BASELINE MODELS

[INFO] Training LogisticRegression...
[INFO] LogisticRegression CV F1-macro: 0.9211 (+/- 0.0198)

[INFO] Training NaiveBayes...
[INFO] NaiveBayes CV F1-macro: 0.9238 (+/- 0.0210)

TRAINING STRONG MODEL

[INFO] Training RandomForest...
[INFO] RandomForest CV F1-macro: 0.9173 (+/- 0.0167)

MODEL EVALUATION ON TEST SET


NotFittedError: This LogisticRegression instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [ ]:
# Error analysis
error_results = classifier.error_analysis(X_test, y_test_sent, df_cleaned)
print(f"\nTotal errors: {error_results['total_errors']}")
print(f"Error rate: {error_results['error_rate']*100:.1f}%")

## 3. Rating Regression

In [ ]:
# Initialize predictor
predictor = RatingPredictor()

# Train baselines
predictor.train_baselines(X_train, y_train_reg)

# Train strong model
predictor.train_strong_model(X_train, y_train_reg)

# Evaluate
reg_results = predictor.evaluate(X_test, y_test_reg)

# Compare
reg_comparison = predictor.compare_models()
print("\nRegression Model Comparison:")
print(reg_comparison)

## 4. Save Results

In [ ]:
# Save models
Path('../outputs/models').mkdir(exist_ok=True)
classifier.save_model('../outputs/models')
predictor.save_model('../outputs/models')

print("Models saved to ../outputs/models/")